In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "PAQ_D.xpt"
csv_file = DATA_DIR / "PAQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched original source
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 PAQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PAD020",
    "PAQ050Q",
    "PAQ050U",
    "PAD080",
    "PAQ100",
    "PAD120",
    "PAD160",
    "PAQ180",
    "PAD200",
    "PAD320",
    "PAD440",
    "PAD460",
    "PAQ500",
    "PAQ520",
    "PAQ540",
    "PAQ560",
    "PAD590",
    "PAD600",
    "PAAQUEX"
]


if len(df) != 9424:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 20:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "PAQ_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: PAQ_D structure = "
    "9,424 rows x 20 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifacts:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. EXPECTED VALIDATED TINY COUNTS
# ============================================================

EXPECTED_TINY_COUNTS = {

    "PAQ560": 79,

    "PAD590": 1252,

    "PAD600": 2699
}


EXPECTED_TINY_TOTAL = 4030


# ============================================================
# 6. VALIDATE TINY ARTIFACT LOCATIONS
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


expected_tiny_columns = set(
    EXPECTED_TINY_COUNTS.keys()
)


unexpected_columns = (
    actual_tiny_columns
    -
    expected_tiny_columns
)


if unexpected_columns:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "PAQ_D variables:\n"
        + str(sorted(unexpected_columns))
    )


missing_expected_columns = (
    expected_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected_columns:

    raise ValueError(
        "STOP: Expected tiny-value variables missing:\n"
        + str(sorted(missing_expected_columns))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts[col]
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


if int(tiny_counts.sum()) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: Exactly 4,030 validated "
    "tiny XPORT artifacts found."
)


# ============================================================
# 7. RESTORE VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<8}: "
        f"{actual_count:,} tiny -> 0"
    )


# ============================================================
# 8. VERIFY RESTORED ZEROS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<8}: "
        f"{zero_count:,}"
    )


    if zero_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} zeros, "
            f"found {zero_count:,}."
        )


print(
    "PASS: All 4,030 validated zeros restored."
)


# ============================================================
# 9. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifact remains."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 10. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# IMPORTANT:
# We do this BEFORE converting anything to Int64.
#
# If any genuine fractional value exists,
# STOP rather than destroy decimal fidelity.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        fractional_columns[col] = count


if fractional_columns:

    print(
        "Fractional variables found:"
    )

    print(
        fractional_columns
    )

    raise ValueError(
        "STOP: Genuine decimal variables found. "
        "Do NOT convert those fields to Int64."
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy()
)


if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    df["SEQN"]
    .isna()
    .sum()
)


seqn_duplicates = int(
    df["SEQN"]
    .duplicated()
    .sum()
)


seqn_min = df["SEQN"].min()
seqn_max = df["SEQN"].max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:
    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:
    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31128:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41474:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 12. CDC CATEGORICAL FREQUENCY CHECKS
# ============================================================

EXPECTED_FREQUENCIES = {

    "PAD020": {
        "counts": {
            1: 2370,
            2: 4644,
            3: 135,
            7: 2
        },
        "missing": 2273
    },

    "PAQ050U": {
        "counts": {
            1: 661,
            2: 1071,
            3: 638
        },
        "missing": 7054
    },

    "PAQ100": {
        "counts": {
            1: 3637,
            2: 2362,
            3: 138,
            7: 1,
            9: 1
        },
        "missing": 3285
    },

    "PAQ180": {
        "counts": {
            1: 1399,
            2: 3120,
            3: 1150,
            4: 464,
            7: 2,
            9: 4
        },
        "missing": 3285
    },

    "PAD200": {
        "counts": {
            1: 3092,
            2: 3816,
            3: 241,
            7: 1,
            9: 1
        },
        "missing": 2273
    },

    "PAD320": {
        "counts": {
            1: 4091,
            2: 2891,
            3: 166,
            7: 1,
            9: 2
        },
        "missing": 2273
    },

    "PAD440": {
        "counts": {
            1: 2574,
            2: 4384,
            3: 190,
            7: 1,
            9: 2
        },
        "missing": 2273
    },

    "PAQ500": {
        "counts": {
            1: 1529,
            2: 1609,
            3: 4009,
            7: 1,
            9: 3
        },
        "missing": 2273
    },

    "PAQ520": {
        "counts": {
            1: 2463,
            2: 1448,
            3: 3164,
            7: 1,
            9: 75
        },
        "missing": 2273
    },

    "PAQ540": {
        "counts": {
            1: 503,
            2: 2467,
            3: 939,
            7: 1,
            9: 2
        },
        "missing": 5512
    },

    "PAD590": {
        "counts": {
            0: 1252,
            1: 1747,
            2: 2556,
            3: 1541,
            4: 902,
            5: 1197,
            6: 145,
            77: 1,
            99: 6
        },
        "missing": 77
    },

    "PAD600": {
        "counts": {
            0: 2699,
            1: 1437,
            2: 791,
            3: 304,
            4: 175,
            5: 236,
            6: 3698,
            77: 1,
            99: 6
        },
        "missing": 77
    },

    "PAAQUEX": {
        "counts": {
            1: 8335,
            2: 1089
        },
        "missing": 0
    }
}


print(
    "\n--- CDC CATEGORICAL FREQUENCY CHECKS ---"
)


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: CDC frequency mismatch.\n"
            f"Expected: {spec['counts']}\n"
            f"Actual:   {actual_counts}"
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: missing count mismatch.\n"
            f"Expected: {spec['missing']:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col:<8} "
        f"valid={sum(actual_counts.values()):,} "
        f"missing={actual_missing:,}"
    )


print(
    "PASS: Categorical frequencies match CDC."
)


# ============================================================
# 13. CDC RANGE CHECK - PAQ050Q
# ============================================================

#
# Number times walked/bicycled:
#
# 1-100 = 2,370
# Missing = 7,054
#

s = df["PAQ050Q"]


if not (
    int(
        s.between(
            1,
            100
        ).sum()
    ) == 2370

    and

    int(
        s.isna()
        .sum()
    ) == 7054
):

    raise ValueError(
        "PAQ050Q differs from CDC."
    )


print(
    "\nPASS: PAQ050Q matches CDC."
)


# ============================================================
# 14. CDC RANGE CHECK - PAD080
# ============================================================

#
# Minutes walked/bicycled per day:
#
# 1-600 = 2,363
# Missing = 7,061
#

s = df["PAD080"]


if not (
    int(
        s.between(
            1,
            600
        ).sum()
    ) == 2363

    and

    int(
        s.isna()
        .sum()
    ) == 7061
):

    raise ValueError(
        "PAD080 differs from CDC."
    )


print(
    "PASS: PAD080 matches CDC."
)


# ============================================================
# 15. CDC RANGE CHECK - PAD120
# ============================================================

#
# CDC already converted this to number of
# times over the past 30 days.
#
# 1-120 = 3,633
# 99999 = 2
# Missing = 5,789
#
# DO NOT recompute.
#

s = df["PAD120"]


if not (
    int(
        s.between(
            1,
            120
        ).sum()
    ) == 3633

    and

    int(
        s.eq(99999)
        .sum()
    ) == 2

    and

    int(
        s.isna()
        .sum()
    ) == 5789
):

    raise ValueError(
        "PAD120 differs from CDC."
    )


print(
    "PASS: PAD120 matches CDC."
)


# ============================================================
# 16. CDC RANGE CHECK - PAD160
# ============================================================

#
# CDC already converted duration to minutes.
#
# 10-600 = 3,624
# Missing = 5,800
#
# DO NOT recalculate.
#

s = df["PAD160"]


if not (
    int(
        s.between(
            10,
            600
        ).sum()
    ) == 3624

    and

    int(
        s.isna()
        .sum()
    ) == 5800
):

    raise ValueError(
        "PAD160 differs from CDC."
    )


print(
    "PASS: PAD160 matches CDC."
)


# ============================================================
# 17. CDC RANGE CHECK - PAD460
# ============================================================

#
# Muscle-strengthening frequency:
#
# 1-210 = 2,573
# 999   = 1
# Missing = 6,850
#

s = df["PAD460"]


if not (
    int(
        s.between(
            1,
            210
        ).sum()
    ) == 2573

    and

    int(
        s.eq(999)
        .sum()
    ) == 1

    and

    int(
        s.isna()
        .sum()
    ) == 6850
):

    raise ValueError(
        "PAD460 differs from CDC."
    )


print(
    "PASS: PAD460 matches CDC."
)


# ============================================================
# 18. CDC CHECK - PAQ560
# ============================================================

#
# Times per week play/exercise hard.
#
# CDC instruction:
# IF NEVER, ENTER 0.
#
# 0-77  = 2,192
# 99999 = 4
# Missing = 7,228
#
# Actual restored zeros = 79
#

s = df["PAQ560"]


if not (
    int(
        s.between(
            0,
            77
        ).sum()
    ) == 2192

    and

    int(
        s.eq(0)
        .sum()
    ) == 79

    and

    int(
        s.eq(99999)
        .sum()
    ) == 4

    and

    int(
        s.isna()
        .sum()
    ) == 7228
):

    raise ValueError(
        "PAQ560 differs from CDC/source."
    )


print(
    "PASS: PAQ560 matches CDC/source."
)

print(
    "      Restored zero / Never = 79"
)


# ============================================================
# 19. CDC CHECK - PAD590
# ============================================================

#
# TV/video hours:
#
# 0 = Less than 1 hour = 1,252
#
# Full distribution already checked above.
#

if int(
    df["PAD590"]
    .eq(0)
    .sum()
) != 1252:

    raise ValueError(
        "PAD590 restored zero count differs."
    )


print(
    "PASS: PAD590 zero count = 1,252."
)


# ============================================================
# 20. CDC CHECK - PAD600
# ============================================================

#
# Computer hours:
#
# 0 = Less than 1 hour = 2,699
#

if int(
    df["PAD600"]
    .eq(0)
    .sum()
) != 2699:

    raise ValueError(
        "PAD600 restored zero count differs."
    )


print(
    "PASS: PAD600 zero count = 2,699."
)


# ============================================================
# 21. EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# Expected output =
#
# original XPT
# +
# ONLY:
#
# PAQ560 tiny -> 0
# PAD590 tiny -> 0
# PAD600 tiny -> 0
#
# Do NOT:
#
# - round
# - recalculate
# - derive new values
# - alter released CDC values
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 22. CLEAN WHOLE-NUMBER FORMATTING
# ============================================================

#
# We already proved genuine fractional variables = 0.
#
# Therefore all PAQ_D variables can safely use Int64.
#
# Examples:
#
# SEQN:
# 31128.0 -> 31128
#
# PAQ050Q:
# 5.0 -> 5
#
# PAQ560:
# 0.0 -> 0
#
# PAD590:
# 0.0 -> 0
#
# Missing:
# NaN -> blank
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and PAQ_D fields "
    "converted to clean integer formatting."
)


# ============================================================
# 23. PRE-EXPORT EXACT VALUE FIDELITY
# ============================================================

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: Values match expected XPT "
    "after ONLY the three validated zero restorations."
)


# ============================================================
# 24. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format
# NO round()
# NO recalculation
#
# Missing SAS values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 25. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 26. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 27. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- XPT -> CSV NUMERIC FIDELITY ---"
)


numeric_failures = []
total_numeric_differences = 0


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Total numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 20 CSV columns numerically "
    "match expected XPT values."
)


# ============================================================
# 28. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(missing_failures)
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 29. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 9424:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 30. STRICT RAW CSV FORMAT FIDELITY
# ============================================================

#
# Every PAQ_D variable has been proven to be
# a whole-number identifier/code/count field.
#
# Expected:
#
# missing -> ""
#
# otherwise:
#
# str(int(value))
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(raw_examples) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print("\nExamples:")


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "PAQ_D raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV formatting matches expected "
    "whole-number representation."
)


# ============================================================
# 31. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(dot_zero_examples) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 32. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(scientific_examples) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 33. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 34. FINAL ZERO CHECK
# ============================================================

print(
    "\n--- FINAL CSV ZERO CHECK ---"
)


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<8}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: final zero count changed."
        )


print(
    "PASS: All final zero counts remain correct."
)


# ============================================================
# 35. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("PAQ_D CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "PAQ560:",
    79
)

print(
    "PAD590:",
    1252
)

print(
    "PAD600:",
    2699
)

print(
    "TOTAL:",
    EXPECTED_TINY_TOTAL
)


print(
    "\nUnexpected tiny variables:",
    0
)

print(
    "Genuine decimal variables:",
    0
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Values ending .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "PAQ_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 PAQ_D XPT -> CSV
Rows: 9,424
Columns: 20
PASS: PAQ_D structure = 9,424 rows x 20 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifacts: 4,030

Tiny values by variable:
PAD600    2699
PAD590    1252
PAQ560      79

PASS: Exactly 4,030 validated tiny XPORT artifacts found.

--- ZERO RESTORATION ---
PAQ560  : 79 tiny -> 0
PAD590  : 1,252 tiny -> 0
PAD600  : 2,699 tiny -> 0

--- RESTORED ZERO COUNTS ---
PAQ560  : 79
PAD590  : 1,252
PAD600  : 2,699
PASS: All 4,030 validated zeros restored.

Remaining tiny artifacts: 0
PASS: No tiny-value artifacts remain.

--- FRACTIONAL VALUE CHECK ---
PASS: Genuine decimal variables = 0.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
Minimum: 31128.0
Maximum: 41474.0
PASS: SEQN validated.

--- CDC CATEGORICAL FREQUENCY CHECKS ---
PASS: PAD020   valid=7,151 missing=2,273
PASS: PAQ050U  valid=2,370 missing=7,054
PASS: PAQ100   valid=6,139 missing=3,285
PASS: PAQ180   valid=6,139 missing=3,285
PASS: PAD200   valid=7,15